In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [ ]:
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
df = pd.read_excel("project_dataset.xlsx")
X = df.drop(columns=["Country", "BiodiversityIndex"])
y = df["BiodiversityIndex"].values
countries = df["Country"]

In [ ]:
scaler = StandardScaler()
#scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
X_train, X_test, y_train, y_test, countries_train, countries_test = train_test_split(
    X_scaled, y, countries, test_size=0.2, random_state=42
)

In [ ]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.reshape(-1, 1), dtype=torch.float32)

In [ ]:
class DeepNN(nn.Module):
    def __init__(self, input_dim):
        super(DeepNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
model = DeepNN(X_train.shape[1])
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
for epoch in range(300):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()

In [ ]:
model.eval()
with torch.no_grad():
    y_pred = model(X_test_tensor).numpy().flatten()
    y_pred = np.clip(y_pred, 0, 1)  # clip between 0-1

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

In [ ]:
print("\n✅ FINAL MODEL RESULTS (All Features Used, Seed Fixed)")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R² Score: {r2:.4f}")

In [ ]:
# === 1. Actual vs Predicted Plot ===
x = np.arange(len(countries_test))
height = 0.35

plt.figure(figsize=(10, 16))
plt.barh(x - height/2, y_test, height, label='Actual', color='seagreen', alpha=0.8)
plt.barh(x + height/2, y_pred, height, label='Predicted', color='skyblue', alpha=0.8)

for i, v in enumerate(y_test):
    plt.text(v + 0.01, i - height/2, f"{v:.4f}", va='center', fontsize=8)
for i, v in enumerate(y_pred):
    plt.text(v + 0.01, i + height/2, f"{v:.4f}", va='center', fontsize=8)

plt.yticks(x, countries_test, fontsize=8)
plt.xlabel("Biodiversity Index")
plt.title("Actual vs Predicted Biodiversity Index (DNN)")
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# === 2. Absolute Error per Country ===
error = np.abs(y_test - y_pred)

plt.figure(figsize=(18, 7))
plt.bar(countries_test, error, color="salmon")
for i, v in enumerate(error):
    plt.text(i, v + 0.005, f"{v:.4f}", color='black', ha='center', fontsize=8)

plt.xticks(rotation=90)
plt.ylabel("Absolute Error")
plt.xlabel("Country")
plt.title("Prediction Error (Absolute Error, Test Countries) - DNN")
plt.tight_layout()
plt.show()

In [ ]:
# === 3. Feature "Importance" via Weight Magnitude (approximation) ===
feature_names = X.columns
with torch.no_grad():
    layer_weights = model.net[0].weight.abs().sum(axis=0).numpy()
    importances = layer_weights / layer_weights.sum()

indices = np.argsort(importances)[::-1]
features_sorted = feature_names[indices]

plt.figure(figsize=(15, 12))
plt.barh(features_sorted, importances[indices], color="steelblue")
for i, v in enumerate(importances[indices]):
    plt.text(v + 0.001, i, f"{v:.5f}", color='black', va='center', fontsize=8)

plt.title("Feature Importance (Approx.) - DNN (by Input Layer Weights)")
plt.xlabel("Normalized Magnitude")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


In [ ]:
# === 4. Grouped Feature Importance ===
category_scores = {
    'Economy': sum(importances[[feature_names.get_loc(f) for f in ['GDP_PPP', 'HDI', 'Surfacearea']]]),
    'Natural Environment': sum(importances[[feature_names.get_loc(f) for f in ['Latitude', 'Forestarea(%)', 'Waterresources(%)']]]),
    'Climate': sum(importances[[i for i, f in enumerate(feature_names) if f not in ['GDP_PPP', 'HDI', 'Surfacearea', 'Latitude', 'Forestarea(%)', 'Waterresources(%)']]])
}

df_grouped = pd.DataFrame.from_dict(category_scores, orient='index', columns=['Importance'])

labels = df_grouped.index
sizes = df_grouped['Importance']
colors = ['#66c2a5', '#fc8d62', '#8da0cb']

plt.figure(figsize=(6, 6))
plt.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=140)
plt.title("Grouped Feature Importance by Category (DNN)")
plt.axis('equal')
plt.tight_layout()
plt.show()